# 08 — Heatmap Visualization

Visualizes ML predictions as a heatmap overlaid on the Manhattan grid.

**Input:** `csv/07_predictions.csv`

**Output:**
- `outputs/latest/08_heatmap_predictions.png` — static matplotlib heatmap
- `outputs/latest/08_heatmap_interactive.html` — interactive folium Leaflet.js map

In [ ]:
# ── Papermill parameters ──────────────────────────────
PLOTS_DIR = "outputs/latest"

In [ ]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import json
import os

os.makedirs(PLOTS_DIR, exist_ok=True)

with open("grid.json", encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]

# Load predictions
df = pd.read_csv("csv/07_predictions.csv", dtype={"cell_id": str})
print(f"Loaded {len(df)} cell predictions")
print(f"Commercial: {(df['predicted_zone'] == 'Commercial').sum()}")
print(f"Residential: {(df['predicted_zone'] == 'Residential').sum()}")

In [ ]:
# ── Compute cell rectangle dimensions in degrees ─────
import math

REF_LAT = df["cell_lat"].mean()
LAT_STEP = CELL_SIZE_M / 111_000
LON_STEP = CELL_SIZE_M / (111_000 * math.cos(math.radians(REF_LAT)))
HALF_LAT = LAT_STEP / 2
HALF_LON = LON_STEP / 2

print(f"Cell size: {LAT_STEP:.6f} lat x {LON_STEP:.6f} lon")
print(f"Lat range: {df['cell_lat'].min():.4f} - {df['cell_lat'].max():.4f}")
print(f"Lon range: {df['cell_lon'].min():.4f} - {df['cell_lon'].max():.4f}")

In [ ]:
# ── Static matplotlib heatmap with city basemap ──────
import contextily as ctx

# Custom diverging colormap: blue (Residential) ↔ red (Commercial)
cmap = LinearSegmentedColormap.from_list(
    "res_com", ["#2166AC", "#D1E5F0", "#FDDBC7", "#B2182B"], N=256)

fig, ax = plt.subplots(figsize=(8, 18))

for _, row in df.iterrows():
    rect = mpatches.Rectangle(
        (row["cell_lon"] - HALF_LON, row["cell_lat"] - HALF_LAT),
        LON_STEP, LAT_STEP,
        linewidth=0.1, edgecolor="gray",
        facecolor=cmap(row["prob_commercial"]),
        alpha=0.75,
    )
    ax.add_patch(rect)

# Set bounds with padding
pad = 0.005
ax.set_xlim(df["cell_lon"].min() - pad, df["cell_lon"].max() + pad)
ax.set_ylim(df["cell_lat"].min() - pad, df["cell_lat"].max() + pad)
ax.set_aspect("equal")

# Add subtle city basemap (streets without labels)
try:
    ctx.add_basemap(ax, crs="EPSG:4326",
                    source=ctx.providers.CartoDB.PositronNoLabels,
                    zoom=14, alpha=0.4)
except Exception as e:
    print(f"Basemap download failed (needs internet): {e}")

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

# Compute percentages for title
n_com = (df["predicted_zone"] == "Commercial").sum()
n_res = (df["predicted_zone"] == "Residential").sum()
pct_com = 100 * n_com / len(df)
pct_res = 100 * n_res / len(df)
ax.set_title(
    f"Manhattan Grid — Commercial Probability\n"
    f"{len(df)} cells ({CELL_SIZE_M}m) · "
    f"Commercial {pct_com:.1f}% · Residential {pct_res:.1f}%",
    fontsize=13)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
cbar.set_label("P(Commercial)", fontsize=11)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.set_ticklabels(["Residential", "0.25", "0.50", "0.75", "Commercial"])

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/08_heatmap_predictions.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/08_heatmap_predictions.png")

In [ ]:
# ── Interactive folium map ─────────────────────────────
try:
    import folium
    from folium import Rectangle
    import branca.colormap as bcm

    m = folium.Map(
        location=[REF_LAT, df["cell_lon"].mean()],
        zoom_start=13,
        tiles="CartoDB positron",
    )

    # Color scale
    colormap = bcm.LinearColormap(
        colors=["#2166AC", "#D1E5F0", "#FDDBC7", "#B2182B"],
        vmin=0, vmax=1,
        caption="P(Commercial)"
    )

    for _, row in df.iterrows():
        bounds = [
            [row["cell_lat"] - HALF_LAT, row["cell_lon"] - HALF_LON],
            [row["cell_lat"] + HALF_LAT, row["cell_lon"] + HALF_LON],
        ]
        color = colormap(row["prob_commercial"])
        popup_text = (f"<b>{row['cell_id']}</b><br>"
                      f"Actual: {row['zone_type']}<br>"
                      f"Predicted: {row['predicted_zone']}<br>"
                      f"P(Commercial): {row['prob_commercial']:.2f}")
        Rectangle(
            bounds=bounds,
            color="gray", weight=0.3,
            fill=True, fill_color=color, fill_opacity=0.75,
            popup=folium.Popup(popup_text, max_width=200),
        ).add_to(m)

    colormap.add_to(m)

    html_path = f"{PLOTS_DIR}/08_heatmap_interactive.html"
    m.save(html_path)
    print(f"Saved: {html_path}")
    print("Open in browser for interactive exploration.")

except ImportError:
    print("folium not installed — skipping interactive map.")
    print("Install with: pip install folium")

In [ ]:
# ── Summary dashboard plot ────────────────────────────

_map = {"Commercial": "Commercial", "Mixed-Use": "Residential", "Residential": "Residential"}
df["actual_binary"] = df["zone_type"].map(_map)

n_total = len(df)
n_com_pred = (df["predicted_zone"] == "Commercial").sum()
n_res_pred = (df["predicted_zone"] == "Residential").sum()
n_com_actual = (df["actual_binary"] == "Commercial").sum()
n_res_actual = (df["actual_binary"] == "Residential").sum()
accuracy = (df["predicted_zone"] == df["actual_binary"]).mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    f"Grid Finding — Summary Dashboard\n"
    f"{n_total} cells · {CELL_SIZE_M}m grid · Overall accuracy: {accuracy:.1%}",
    fontsize=14, fontweight="bold")

# (0,0) Predicted class distribution — pie chart
ax = axes[0, 0]
sizes = [n_com_pred, n_res_pred]
labels = [f"Commercial\n{n_com_pred} ({100*n_com_pred/n_total:.1f}%)",
          f"Residential\n{n_res_pred} ({100*n_res_pred/n_total:.1f}%)"]
colors = ["#B2182B", "#2166AC"]
ax.pie(sizes, labels=labels, colors=colors, autopct="", startangle=90,
       textprops={"fontsize": 11}, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax.set_title("Predicted Distribution", fontsize=12)

# (0,1) Actual vs Predicted — grouped bar chart
ax = axes[0, 1]
x = np.arange(2)
width = 0.35
bars_actual = [n_com_actual, n_res_actual]
bars_pred = [n_com_pred, n_res_pred]
ax.bar(x - width/2, bars_actual, width, label="Actual (PLUTO)", color=["#B2182B", "#2166AC"], alpha=0.6)
ax.bar(x + width/2, bars_pred, width, label="Predicted (ML)", color=["#B2182B", "#2166AC"], alpha=1.0)
ax.set_xticks(x)
ax.set_xticklabels(["Commercial", "Residential"])
ax.set_ylabel("Cell count")
ax.set_title("Actual vs Predicted", fontsize=12)
ax.legend()
for i, (a, p) in enumerate(zip(bars_actual, bars_pred)):
    ax.text(i - width/2, a + 10, str(a), ha="center", fontsize=9)
    ax.text(i + width/2, p + 10, str(p), ha="center", fontsize=9)

# (1,0) Probability distribution histogram
ax = axes[1, 0]
ax.hist(df["prob_commercial"], bins=30, color="#888888", edgecolor="white", alpha=0.8)
ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="Decision boundary")
ax.set_xlabel("P(Commercial)")
ax.set_ylabel("Cell count")
ax.set_title("Probability Distribution", fontsize=12)
ax.legend()
# Annotate median
median_p = df["prob_commercial"].median()
ax.axvline(median_p, color="#2166AC", linestyle=":", linewidth=1)
ax.text(median_p + 0.02, ax.get_ylim()[1] * 0.9, f"median={median_p:.2f}",
        color="#2166AC", fontsize=9)

# (1,1) Key feature means by predicted class
ax = axes[1, 1]
key_feats = ["amenity_density", "shop_density_km2", "tourism_density",
             "brand_ratio", "landuse_entropy"]
available_feats = [f for f in key_feats if f in df.columns]
if available_feats:
    means = df.groupby("predicted_zone")[available_feats].mean()
    means_t = means.T
    x = np.arange(len(available_feats))
    if "Commercial" in means_t.columns and "Residential" in means_t.columns:
        ax.barh(x - 0.2, means_t["Commercial"], 0.35, label="Commercial", color="#B2182B", alpha=0.85)
        ax.barh(x + 0.2, means_t["Residential"], 0.35, label="Residential", color="#2166AC", alpha=0.85)
    ax.set_yticks(x)
    ax.set_yticklabels(available_feats, fontsize=9)
    ax.set_xlabel("Mean value")
    ax.set_title("Feature Means by Predicted Class", fontsize=12)
    ax.legend()

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/09_summary_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {PLOTS_DIR}/09_summary_dashboard.png")

In [ ]:
# ── Summary statistics ────────────────────────────────
print("Prediction summary:")
print(f"  Total cells: {len(df)}")
print(f"  Predicted Commercial: {n_com_pred} ({pct_com:.1f}%)")
print(f"  Predicted Residential: {n_res_pred} ({pct_res:.1f}%)")
print(f"  Mean P(Commercial): {df['prob_commercial'].mean():.3f}")
print(f"  Median P(Commercial): {df['prob_commercial'].median():.3f}")
print(f"\n  Overall accuracy (on all cells): {accuracy:.3f}")